## Import de librerías

In [1]:
# Import de librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import statsmodels.api as sm
import glob
import polars as pl
import timeit
from category_encoders import HashingEncoder

## Import de dataset

In [2]:
# path a los archivos parquet
##files = glob.glob("../dataset/*.parquet")
files = glob.glob("../dataset/02-preprocessed/yellow_tripdata_2024-01.parquet")

# leer .parquet files y concatenar en un solo df 
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print(len(files), "files merged")
print(df.shape)

# Formato de visualización de floats en pandas
pd.set_option('display.float_format', '{:.2f}'.format)

1 files merged
(2646948, 24)


In [3]:
df.head(5)
df.describe()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,month
count,2646948,2646948,2520390.00,2646948.00,2646948.00,2646948.00,2646948.00,2646948.00,2646948.00,2646948.00,2646948.00,2520390.00,2520390.00,2646948.00
mean,2024-01-17 03:30:29.999934,2024-01-17 03:44:13.376606,1.34,2.66,14.71,1.30,0.49,2.86,0.21,0.98,22.38,2.38,0.00,1.00
min,2002-12-31 22:59:39,2002-12-31 23:05:41,0.00,0.00,-744.30,-7.50,-0.50,-80.00,-80.00,-1.00,-753.74,-2.50,-1.75,1.00
25%,2024-01-09 18:09:14,2024-01-09 18:22:02,1.00,0.94,8.60,0.00,0.50,1.00,0.00,1.00,15.02,2.50,0.00,1.00
50%,2024-01-17 13:53:15,2024-01-17 14:09:59.500000,1.00,1.53,12.10,1.00,0.50,2.66,0.00,1.00,19.11,2.50,0.00,1.00
75%,2024-01-24 19:50:11,2024-01-24 20:03:13.250000,1.00,2.54,17.00,2.50,0.50,3.86,0.00,1.00,25.42,2.50,0.00,1.00
max,2024-02-01 00:01:15,2024-02-01 22:04:34,9.00,312722.30,761.10,12.50,4.00,303.00,95.46,1.00,775.48,2.50,1.75,12.00
std,NaN,NaN,0.85,227.68,12.06,1.51,0.11,2.86,1.49,0.21,15.05,0.64,0.02,0.02


**Variables Categóricas**

In [4]:
# convertir objects a categoricas
categorical_cols = ['vendor', 'ratecode', 'payment_type', 'pickup_borough', 'pickup_zone', 'pickup_service_zone', 'dropoff_borough', 'dropoff_zone', 'dropoff_service_zone']
for col in categorical_cols:
    df[col] = df[col].astype('category')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2646948 entries, 0 to 2646947
Data columns (total 24 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   vendor                 category      
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        Int64         
 4   trip_distance          float64       
 5   ratecode               category      
 6   store_and_fwd_flag     boolean       
 7   payment_type           category      
 8   fare_amount            float64       
 9   extra                  float64       
 10  mta_tax                float64       
 11  tip_amount             float64       
 12  tolls_amount           float64       
 13  improvement_surcharge  float64       
 14  total_amount           float64       
 15  congestion_surcharge   float64       
 16  Airport_fee            float64       
 17  pickup_borough         category      
 18  pickup_zone           

In [5]:
df.describe(include='category')

,vendor,ratecode,payment_type,pickup_borough,pickup_zone,pickup_service_zone,dropoff_borough,dropoff_zone,dropoff_service_zone
count,2646948,2512036,2646948,2646948,2646948,2646948,2644100,2639063,2636215
unique,2,6,5,1,67,2,7,257,4
top,"Curb Mobility, LLC",Standard rate,Credit card,Manhattan,Midtown Center,Yellow Zone,Manhattan,Upper East Side North,Yellow Zone
freq,1996347,2469767,2086997,2646948,143471,2591698,2492571,137411,2373086


Eliminamos columnas que no vamos a usar

In [6]:
cols_delete = ['vendor', 'passenger_count', 'ratecode','store_and_fwd_flag', 'payment_type', 'pickup_service_zone', 'dropoff_borough', 'dropoff_service_zone', 'dropoff_zone', 'extra', 'mta_tax', 'improvement_surcharge', 'tolls_amount', 'congestion_surcharge', 'Airport_fee', 'total_amount']
df = df.drop(columns=cols_delete)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2646948 entries, 0 to 2646947
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[us]
 1   tpep_dropoff_datetime  datetime64[us]
 2   trip_distance          float64       
 3   fare_amount            float64       
 4   tip_amount             float64       
 5   pickup_borough         category      
 6   pickup_zone            category      
 7   month                  int32         
dtypes: category(2), datetime64[us](2), float64(3), int32(1)
memory usage: 116.1 MB


In [8]:
df['pickup_zone'].value_counts()

pickup_zone
Midtown Center                                   143471
Upper East Side South                            142708
Upper East Side North                            136465
Midtown East                                     106717
Times Sq/Theatre District                        106324
                                                  ...  
Randalls Island                                      98
Marble Hill                                          49
Inwood Hill Park                                     22
Highbridge Park                                      15
Governor's Island/Ellis Island/Liberty Island         1
Name: count, Length: 67, dtype: int64

Variable target = pickup_zone
Debido a que son muchas categorías, decidimos aplicar Hashing Encoding.
Si quisieramos poder representar las 67 categorías del dataset de entrenamiento, necesitariamos 7 columnas
Dado que es un dtaset con pocas columnas, no habria un gran impacto en agregar 7 columnas extras para el hashing, y evitaríamos colisiones. Con 7 columnas podemos representar hasta 127 categorías


In [9]:
# 5. Hashing Encoding
hashing_encoder = HashingEncoder(n_components=7) 
# Nota: n_components es el nro de columnas a generar. Si hay más, hay menos riesgo de colisión 
# pero mayor dimensionalidad.

hash_encoded = hashing_encoder.fit_transform(df[["pickup_zone"]])
df = df.join(hash_encoded)
df.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,tip_amount,pickup_borough,pickup_zone,month,col_0,col_1,col_2,col_3,col_4,col_5,col_6
0,2024-01-01 00:57:55,2024-01-01 01:17:43,1.72,17.70,0.00,Manhattan,Penn Station/Madison Sq West,1,0,1,0,0,0,0,0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.80,10.00,3.75,Manhattan,Lenox Hill East,1,0,1,0,0,0,0,0
2,2024-01-01 00:17:06,2024-01-01 00:35:01,4.70,23.30,3.00,Manhattan,Upper East Side North,1,0,0,0,0,0,1,0
3,2024-01-01 00:36:38,2024-01-01 00:44:56,1.40,10.00,2.00,Manhattan,East Village,1,0,0,0,0,0,0,1
4,2024-01-01 00:46:51,2024-01-01 00:52:57,0.80,7.90,3.20,Manhattan,SoHo,1,0,0,0,1,0,0,0


In [10]:
df.shape

(2646948, 15)